In [4]:
print("starting")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import neurokit2 as nk
from scipy.stats import skew, kurtosis
from scipy.interpolate import interp1d
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GroupKFold, LeaveOneOut, LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import xgboost as xgb
import tsfel


try:
    import catboost as cb
    has_catboost = True
except ImportError:
    has_catboost = False

try:
    import lightgbm as lgb
    has_lightgbm = True
except ImportError:
    has_lightgbm = False

# Set style for modern aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 150

csv_path = "data/real/glucosense_r_dataset_2026-09-17.csv"
sampling_rate = 50  # 50 Hz acquisition rate
print(f"✓ Imports completed. CatBoost: {has_catboost}, LightGBM: {has_lightgbm}, XGBoost: True.")

starting
✓ Imports completed. CatBoost: True, LightGBM: True, XGBoost: True.


In [8]:
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset file not found at {csv_path}")

df = pd.read_csv(csv_path)
print(f"✓ Dataset loaded. Total records: {len(df)}")
print("\nFirst 3 rows:")
display(df.head(3))

print("\nSession kinds distribution:")
print(df['session_kind'].value_counts())

✓ Dataset loaded. Total records: 100

First 3 rows:


,session_id,created_at,participant_id,age,gender,weight_kg,height_cm,years_on_t2dm,medication,session_kind,bgl_mgdl,ppg_samples_count,ppg_data
0,fce4b7a2-b3db-4b11-81e7-9c8f8ace228d,2026-08-06T09:03:50.335985+00:00,P001,73,Female,74.0,170.0,8,"Alphabetuc+, glepid-2, sealmet, trevisa",fasting,196,3500,57994;58030;58020;58027;58013;58039;58044;5805...
1,07dc95d8-8001-417f-b139-d4faf092b371,2026-08-06T10:34:31.431381+00:00,P001,73,Female,74.0,170.0,8,"Alphabetuc+, glepid-2, sealmet, trevisa",1hr_post_prandial,269,3500,59419;59453;59470;59475;59513;59507;59529;5956...
2,2fb9eb9e-4f36-4749-bb66-78510db5dde9,2026-08-27T07:56:02.871753+00:00,P002,63,Female,79.0,172.0,21,NaN,fasting,190,3500,55490;55479;55469;55480;55473;55469;55476;5548...



Session kinds distribution:
session_kind
fasting              46
2hr_post_prandial    31
1hr_post_prandial    23
Name: count, dtype: int64


In [5]:
def nano_segment_and_average(ppg_raw, sampling_rate=50, target_len=100):
    """
    Adapted implementation of the preprocessing, heartbeat segmentation,
    and heartbeat averaging procedure described in Adigüzel et al. (2024).

    Steps:
    1. Clean PPG using NeuroKit2 / Elgendi.
    2. Detect PPG peaks.
    3. Identify heartbeat segments.
    4. Resample beats to a common length.
    5. Average the valid beats.

    Note:
    The paper does not provide enough detail to establish its exact
    internal segmentation/resampling implementation, so this is an
    explicit reconstruction of the documented methodology rather than
    a claim of byte-for-byte replication.
    """

    # ---------------------------------------------------------
    # 1. PPG preprocessing
    # ---------------------------------------------------------
    cleaned = nk.ppg_clean(
        ppg_raw,
        sampling_rate=sampling_rate,
        method="elgendi"
    )

    # ---------------------------------------------------------
    # 2. Peak detection
    # ---------------------------------------------------------
    _, info = nk.ppg_peaks(
        cleaned,
        sampling_rate=sampling_rate,
        method="elgendi"
    )

    peaks = info["PPG_Peaks"]

    if len(peaks) < 3:
        return None, 0

    # ---------------------------------------------------------
    # 3. Segment between consecutive peaks
    #
    # The paper states that segmentation was based on
    # heartbeat peaks. The exact boundary algorithm is not
    # fully specified, so we use consecutive peak intervals.
    # ---------------------------------------------------------
    beats = []

    for i in range(len(peaks) - 1):

        start = peaks[i]
        end = peaks[i + 1]

        beat = cleaned[start:end]

        # Reject obviously implausible segments.
        # At 50 Hz, these correspond to approximately
        # 0.36–1.6 seconds.
        duration = len(beat) / sampling_rate

        if duration < 0.36 or duration > 1.6:
            continue

        if len(beat) < 5:
            continue

        # -----------------------------------------------------
        # Resample each heartbeat to common length
        # -----------------------------------------------------
        x_old = np.linspace(0, 1, len(beat))
        x_new = np.linspace(0, 1, target_len)

        try:
            beat_resampled = interp1d(
                x_old,
                beat,
                kind="linear"
            )(x_new)
        except Exception:
            continue

        # -----------------------------------------------------
        # Beat-wise amplitude normalisation
        #
        # This is an implementation choice because the paper
        # does not specify the exact normalisation procedure.
        # It prevents amplitude differences from dominating
        # the average morphology.
        # -----------------------------------------------------
        b_min = np.min(beat_resampled)
        b_max = np.max(beat_resampled)

        if (b_max - b_min) < 1e-8:
            continue

        beat_normalised = (
            (beat_resampled - b_min) /
            (b_max - b_min)
        )

        beats.append(beat_normalised)

    if len(beats) == 0:
        return None, 0

    beats = np.asarray(beats)

    # ---------------------------------------------------------
    # 4. Average segmented beats
    # ---------------------------------------------------------
    avg_beat = np.mean(beats, axis=0)

    return avg_beat, len(beats)

In [13]:
# Load TSFEL's default configuration
nano_cfg = tsfel.get_features_by_domain()

print(nano_cfg.keys())

dict_keys(['spectral', 'statistical', 'temporal', 'fractal'])


In [14]:
def extract_nano_tsfel_features(avg_beat, cfg):
    """
    Extract TSFEL statistical and temporal features
    from the averaged PPG heartbeat.
    """

    if avg_beat is None:
        return None

    # TSFEL expects a 1-D time series.
    signal = np.asarray(avg_beat, dtype=float)

    # TSFEL feature extraction
    features = tsfel.time_series_features_extractor(
        cfg,
        signal,
        fs=50,
        verbose=0
    )

    # Convert one-row dataframe to dictionary
    return features.iloc[0].to_dict()

In [15]:
features_list = []
skipped_rows = 0

print("Running NANO-style preprocessing + heartbeat averaging + TSFEL...")
print("-" * 70)

for idx, row in df.iterrows():

    # ---------------------------------------------------------
    # Check required values
    # ---------------------------------------------------------
    if pd.isna(row["bgl_mgdl"]) or pd.isna(row["ppg_data"]):
        skipped_rows += 1
        continue

    try:

        # -----------------------------------------------------
        # Convert stored PPG string to numerical array
        # -----------------------------------------------------
        ppg_signal = np.array(
            [
                float(x)
                for x in str(row["ppg_data"]).split(";")
                if x.strip()
            ],
            dtype=float
        )

        # Expected original recording:
        # 70 seconds × 50 Hz = 3500 samples
        if len(ppg_signal) < 3500:
            skipped_rows += 1
            continue

        # -----------------------------------------------------
        # Remove first 10 seconds
        #
        # 10 sec × 50 Hz = 500 samples
        # -----------------------------------------------------
        active_signal = ppg_signal[500:3500]

        # -----------------------------------------------------
        # Segment + average heartbeat
        # -----------------------------------------------------
        avg_beat, num_beats = nano_segment_and_average(
            active_signal,
            sampling_rate=50,
            target_len=100
        )

        if avg_beat is None:
            skipped_rows += 1
            continue

        # -----------------------------------------------------
        # TSFEL features
        # -----------------------------------------------------
        feat_dict = extract_nano_tsfel_features(
            avg_beat,
            nano_cfg
        )

        if feat_dict is None:
            skipped_rows += 1
            continue

        # -----------------------------------------------------
        # Metadata
        # -----------------------------------------------------
        feat_dict["num_beats"] = num_beats
        feat_dict["participant_id"] = row["participant_id"]
        feat_dict["session_kind"] = row["session_kind"]

        # Target
        feat_dict["bgl_mgdl"] = float(row["bgl_mgdl"])

        features_list.append(feat_dict)

    except Exception as e:

        print(f"Skipped row {idx}: {e}")
        skipped_rows += 1


nano_features_df = pd.DataFrame(features_list)

print("\n" + "=" * 70)
print("EXTRACTION COMPLETE")
print("=" * 70)

print(f"Valid sessions : {len(nano_features_df)}")
print(f"Skipped rows   : {skipped_rows}")
print(f"Total columns  : {nano_features_df.shape[1]}")

Running NANO-style preprocessing + heartbeat averaging + TSFEL...
----------------------------------------------------------------------

EXTRACTION COMPLETE
Valid sessions : 100
Skipped rows   : 0
Total columns  : 160


In [16]:
feature_cols = [
    c for c in nano_features_df.columns
    if c not in [
        "participant_id",
        "session_kind",
        "bgl_mgdl",
        "num_beats"
    ]
]

print(f"Number of TSFEL features: {len(feature_cols)}")

for i, feature in enumerate(feature_cols, 1):
    print(f"{i:3}. {feature}")

Number of TSFEL features: 156
  1. 0_Absolute energy
  2. 0_Area under the curve
  3. 0_Autocorrelation
  4. 0_Average power
  5. 0_Centroid
  6. 0_ECDF Percentile Count_0
  7. 0_ECDF Percentile Count_1
  8. 0_ECDF Percentile_0
  9. 0_ECDF Percentile_1
 10. 0_ECDF_0
 11. 0_ECDF_1
 12. 0_ECDF_2
 13. 0_ECDF_3
 14. 0_ECDF_4
 15. 0_ECDF_5
 16. 0_ECDF_6
 17. 0_ECDF_7
 18. 0_ECDF_8
 19. 0_ECDF_9
 20. 0_Entropy
 21. 0_Fundamental frequency
 22. 0_Histogram mode
 23. 0_Human range energy
 24. 0_Interquartile range
 25. 0_Kurtosis
 26. 0_LPCC_0
 27. 0_LPCC_1
 28. 0_LPCC_10
 29. 0_LPCC_11
 30. 0_LPCC_2
 31. 0_LPCC_3
 32. 0_LPCC_4
 33. 0_LPCC_5
 34. 0_LPCC_6
 35. 0_LPCC_7
 36. 0_LPCC_8
 37. 0_LPCC_9
 38. 0_MFCC_0
 39. 0_MFCC_1
 40. 0_MFCC_10
 41. 0_MFCC_11
 42. 0_MFCC_2
 43. 0_MFCC_3
 44. 0_MFCC_4
 45. 0_MFCC_5
 46. 0_MFCC_6
 47. 0_MFCC_7
 48. 0_MFCC_8
 49. 0_MFCC_9
 50. 0_Max
 51. 0_Max power spectrum
 52. 0_Maximum frequency
 53. 0_Mean
 54. 0_Mean absolute deviation
 55. 0_Mean absolute diff
 

# Correlation Analysis

In [28]:
from IPython.display import display
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Identify TSFEL feature columns
# ---------------------------------------------------------
exclude_cols = [
    "participant_id",
    "session_kind",
    "bgl_mgdl",
    "num_beats"
]

feature_cols = [
    c for c in nano_features_df.columns
    if c not in exclude_cols
]

results = []

for feature in feature_cols:

    # Keep only paired valid observations
    temp = nano_features_df[[feature, "bgl_mgdl"]].dropna()

    if len(temp) < 3:
        continue

    x = temp[feature].values
    y = temp["bgl_mgdl"].values

    # Skip constant features
    if np.std(x) == 0:
        continue

    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)

    results.append({
        "feature": feature,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "abs_pearson_r": abs(pearson_r),
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "abs_spearman_rho": abs(spearman_rho),
        "n": len(temp)
    })

corr_df = pd.DataFrame(results)

# Sort by absolute Pearson correlation
corr_df = corr_df.sort_values(
    "abs_pearson_r",
    ascending=False
).reset_index(drop=True)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(corr_df)

/tmp/ipykernel_15345/13357293.py:38: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r, pearson_p = pearsonr(x, y)
/tmp/ipykernel_15345/13357293.py:39: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_rho, spearman_p = spearmanr(x, y)
/tmp/ipykernel_15345/13357293.py:38: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r, pearson_p = pearsonr(x, y)
/tmp/ipykernel_15345/13357293.py:39: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_rho, spearman_p = spearmanr(x, y)
/tmp/ipykernel_15345/13357293.py:38: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r, pearson_p = pearsonr(x, y)
/tmp/ipykernel_15345/13357293.py:39: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp

,feature,pearson_r,pearson_p,abs_pearson_r,spearman_rho,spearman_p,abs_spearman_rho,n
0,0_MFCC_8,-0.178413,0.075736,0.178413,-0.193765,0.053405,0.193765,100
1,0_MFCC_2,0.177475,0.077314,0.177475,0.210940,0.035148,0.210940,100
2,0_Spectrogram mean coefficient_1.61Hz,0.173562,0.084179,0.173562,0.293831,0.003006,0.293831,100
3,0_Neighbourhood peaks,0.171870,0.087295,0.171870,0.123887,0.219434,0.123887,100
4,0_Spectrogram mean coefficient_22.58Hz,0.168576,0.093623,0.168576,0.025920,0.797965,0.025920,100
5,0_Signal distance,0.161141,0.109237,0.161141,0.245814,0.013694,0.245814,100
6,0_Wavelet variance_6.25Hz,0.160979,0.109598,0.160979,0.154828,0.124023,0.154828,100
7,0_Spectrogram mean coefficient_8.87Hz,0.159917,0.111990,0.159917,0.136992,0.174110,0.136992,100
8,0_Wavelet standard deviation_6.25Hz,0.158736,0.114699,0.158736,0.154828,0.124023,0.154828,100
9,0_Wavelet variance_4.17Hz,0.158371,0.115545,0.158371,0.207262,0.038538,0.207262,100


In [29]:
corr_df["correlation_difference"] = (
    corr_df["pearson_r"] -
    corr_df["spearman_rho"]
).abs()

display(
    corr_df[
        [
            "feature",
            "abs_pearson_r",
            "pearson_r",
            "spearman_rho",
            "correlation_difference"
        ]
    ].sort_values(
        "abs_pearson_r",
        ascending=False
    ).head(30)
)

TOP 30 FEATURES BY ABSOLUTE PEARSON CORRELATION


,feature,pearson_r,pearson_p,spearman_rho,spearman_p,n
0,0_MFCC_8,-0.178413,0.075736,-0.193765,0.053405,100
1,0_MFCC_2,0.177475,0.077314,0.210940,0.035148,100
2,0_Spectrogram mean coefficient_1.61Hz,0.173562,0.084179,0.293831,0.003006,100
3,0_Neighbourhood peaks,0.171870,0.087295,0.123887,0.219434,100
4,0_Spectrogram mean coefficient_22.58Hz,0.168576,0.093623,0.025920,0.797965,100
5,0_Signal distance,0.161141,0.109237,0.245814,0.013694,100
6,0_Wavelet variance_6.25Hz,0.160979,0.109598,0.154828,0.124023,100
7,0_Spectrogram mean coefficient_8.87Hz,0.159917,0.111990,0.136992,0.174110,100
8,0_Wavelet standard deviation_6.25Hz,0.158736,0.114699,0.154828,0.124023,100
9,0_Wavelet variance_4.17Hz,0.158371,0.115545,0.207262,0.038538,100


In [30]:
corr_02 = corr_df[
    corr_df["abs_pearson_r"] >= 0.20
].copy()

print(f"Features with |Pearson r| >= 0.20: {len(corr_02)}")

display(
    corr_02[
        [
            "feature",
            "pearson_r",
            "pearson_p",
            "spearman_rho",
            "spearman_p"
        ]
    ]
)

Features with |Pearson r| >= 0.20: 0


,feature,pearson_r,pearson_p,spearman_rho,spearman_p


In [31]:
corr_df["correlation_difference"] = (
    corr_df["pearson_r"] -
    corr_df["spearman_rho"]
).abs()

display(
    corr_df[
        [
            "feature",
            "abs_pearson_r",
            "pearson_r",
            "spearman_rho",
            "correlation_difference"
        ]
    ].sort_values(
        "abs_pearson_r",
        ascending=False
    ).head(30)
)

,feature,pearson_r,pearson_p,pearson_p_fdr,pearson_significant_fdr
0,0_MFCC_8,-0.178413,0.075736,NaN,False
1,0_MFCC_2,0.177475,0.077314,NaN,False
2,0_Spectrogram mean coefficient_1.61Hz,0.173562,0.084179,NaN,False
3,0_Neighbourhood peaks,0.171870,0.087295,NaN,False
4,0_Spectrogram mean coefficient_22.58Hz,0.168576,0.093623,NaN,False
5,0_Signal distance,0.161141,0.109237,NaN,False
6,0_Wavelet variance_6.25Hz,0.160979,0.109598,NaN,False
7,0_Spectrogram mean coefficient_8.87Hz,0.159917,0.111990,NaN,False
8,0_Wavelet standard deviation_6.25Hz,0.158736,0.114699,NaN,False
9,0_Wavelet variance_4.17Hz,0.158371,0.115545,NaN,False


In [32]:
corr_df["correlation_difference"] = (
    corr_df["pearson_r"] -
    corr_df["spearman_rho"]
).abs()

display(
    corr_df[
        [
            "feature",
            "abs_pearson_r",
            "pearson_r",
            "spearman_rho",
            "correlation_difference"
        ]
    ].sort_values(
        "abs_pearson_r",
        ascending=False
    ).head(30)
)

,feature,abs_pearson_r,pearson_r,spearman_rho,correlation_difference
0,0_MFCC_8,0.178413,-0.178413,-0.193765,0.015351
1,0_MFCC_2,0.177475,0.177475,0.210940,0.033466
2,0_Spectrogram mean coefficient_1.61Hz,0.173562,0.173562,0.293831,0.120268
3,0_Neighbourhood peaks,0.171870,0.171870,0.123887,0.047983
4,0_Spectrogram mean coefficient_22.58Hz,0.168576,0.168576,0.025920,0.142657
5,0_Signal distance,0.161141,0.161141,0.245814,0.084673
6,0_Wavelet variance_6.25Hz,0.160979,0.160979,0.154828,0.006151
7,0_Spectrogram mean coefficient_8.87Hz,0.159917,0.159917,0.136992,0.022925
8,0_Wavelet standard deviation_6.25Hz,0.158736,0.158736,0.154828,0.003908
9,0_Wavelet variance_4.17Hz,0.158371,0.158371,0.207262,0.048890
